In [11]:
# ===== 1. IMPORTS & CONFIGURATION =====
from openai import OpenAI
from langchain_community.chat_message_histories import ChatMessageHistory
import requests
import time

# FIXED CONFIGURATION - Updated for your setup
MODEL_ID = "google/gemma-3-4b"  # Updated to your model
BASE_URL = "http://127.0.0.1:1234/v1"  # Updated to your URL
API_KEY = "lm-studio"


In [12]:
# ===== 2. LM STUDIO HEALTH CHECK =====
def check_lm_studio():
    """Check if LM Studio is running and model is loaded"""
    try:
        print("Checking LM Studio connection...")
        response = requests.get(f"{BASE_URL}/models", timeout=5)
        if response.status_code == 200:
            models = response.json().get('data', [])
            model_names = [m['id'] for m in models]
            
            if MODEL_ID in model_names:
                print(f"LM Studio running! Model '{MODEL_ID}' loaded")
                return True
            else:
                print(f"Model '{MODEL_ID}' not loaded!")
                print(f"Available models: {model_names}")
                return False
        else:
            print("LM Studio not responding on port 1234")
            return False
    except requests.exceptions.ConnectionError:
        print("LM Studio NOT RUNNING on localhost:1234")
        print("\nQUICK FIX:")
        print("1. Open LM Studio")
        print("2. Load model: 'google/gemma-3-1b'")
        print("3. Click 'Start Server' (port 1234)")
        print("4. Run this script again")
        return False

In [13]:
# ===== 3. INITIALIZATION =====
def initialize_client():
    """Initialize OpenAI client with error handling"""
    try:
        return OpenAI(base_url=BASE_URL, api_key=API_KEY)
    except Exception as e:
        print(f"Client initialization failed: {e}")
        return None


In [14]:
# ===== 4. HELPER FUNCTIONS =====
def print_header():
    print(f"--- Session Started with {MODEL_ID} ---")
    print("Commands: Type 'exit', 'quit', or press Ctrl+C to stop.\n")

def get_user_input():
    return input("User (or 'exit'): ").strip()

def should_exit(user_input):
    return user_input.lower() in ['exit', 'quit']

def is_empty_input(user_input):
    return not user_input

def prepare_chat_context(history):
    return [
        {"role": "user" if m.type == "human" else "assistant", "content": m.content}
        for m in history.messages
    ]

def display_turn(user_input):
    print(f"\n[YOU]: {user_input}")
    print("-" * 60)
    print("\n[Gemma 3]: ", end="", flush=True)

def stream_ai_response(client, model_id, chat_context):
    """Stream AI response with error handling"""
    try:
        stream = client.chat.completions.create(
            model=model_id,
            messages=chat_context,
            stream=True,
            temperature=0.7
        )
        
        full_response = ""
        for chunk in stream:
            content = chunk.choices[0].delta.content
            if content:
                print(content, end="", flush=True)
                full_response += content
        return full_response
    except Exception as e:
        error_msg = f"\nAI Error: {str(e)}"
        print(error_msg)
        return error_msg

def finalize_turn(history, ai_response):
    history.add_ai_message(ai_response)
    print(f"\n{'=' * 60}\n")

In [15]:
# ===== 5. MAIN EXECUTION =====
def main():
    """Main chat loop with full error handling"""
    # Step 1: Check LM Studio
    if not check_lm_studio():
        input("\nPress Enter after starting LM Studio...")
        return
    
    # Step 2: Initialize client
    client = initialize_client()
    if not client:
        return
    
    # Step 3: Initialize history
    history = ChatMessageHistory()
    print_header()
    
    try:
        while True:
            user_input = get_user_input()
            
            if should_exit(user_input):
                break
            
            if is_empty_input(user_input):
                continue
            
            history.add_user_message(user_input)
            chat_context = prepare_chat_context(history)
            display_turn(user_input)
            
            ai_response = stream_ai_response(client, MODEL_ID, chat_context)
            finalize_turn(history, ai_response)
            
    except KeyboardInterrupt:
        print("\n\nInterrupted by user. Closing session...")
    finally:
        print("Program Finished.")

if __name__ == "__main__":
    main()

Checking LM Studio connection...
LM Studio running! Model 'google/gemma-3-4b' loaded
--- Session Started with google/gemma-3-4b ---
Commands: Type 'exit', 'quit', or press Ctrl+C to stop.


[YOU]: I am Aaron Ludwig A. Altar
------------------------------------------------------------

[Gemma 3]: It's nice to meet you, Aaron Ludwig A. Altar! That’s quite a name – it has a really interesting flow to it. 

Is there anything you'd like to tell me about yourself? Perhaps you want to:

*   **Just chat?** We can talk about pretty much anything.
*   **Ask me something?** Do you have a question on your mind?
*   **Tell me a little about yourself?** I'm always interested in learning about people!


[YOU]: Who can you say are the GOAT's of Basketball?
------------------------------------------------------------

[Gemma 3]: Okay, this is *always* a hotly debated topic! Declaring the "GOAT" (Greatest Of All Time) in basketball is incredibly subjective and depends on what criteria you prioritize – s